# Análise Exploratória: Perfil e Dores do Eleitorado

Neste notebook, realizamos a Análise Exploratória de Dados (EDA) primária do dataset de respostas da Bússola Eleitoral.

## Objetivos:
1. Compreender a distribuição demográfica da amostra.
2. Identificar correlações iniciais entre clivagens (Renda, Religião, Idade) e os blocos temáticos.
3. Estabelecer as bases para a Ponderação Amostral (Post-Stratification).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configurações estéticas do Seaborn
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)


## 1. Carregamento dos Dados
Importamos os dados sintéticos gerados pelo módulo de mock_data.


In [ ]:
df = pd.read_csv('../data/raw/mock_voters.csv')
display(df.head())
print(f'Tamanho da amostra: {df.shape[0]} eleitores')


## 2. Clivagens Sociais: Demografia
A teoria das clivagens de Lipset e Rokkan argumenta que traços estruturais moldam o voto. Vamos analisar a composição da nossa amostra.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x='idade', order=['16-24', '25-34', '35-44', '45-59', '60+'], ax=axes[0])
axes[0].set_title('Distribuição por Idade')

sns.countplot(data=df, x='faixa_renda', order=['Ate 2SM', '2 a 5SM', '5 a 10SM', 'Mais de 10SM'], ax=axes[1])
axes[1].set_title('Distribuição por Renda')

sns.countplot(data=df, x='religiao', ax=axes[2])
axes[2].set_title('Distribuição por Religião')

plt.tight_layout()
plt.show()


## 3. Distribuição de Opiniões (Eixos Econômico e Valores)
Analisamos agora as respostas (em escala Likert de 1 a 5) sobre o papel do Estado (Bloco D) e Valores/Punitivismo (Bloco E).


In [ ]:
cols_opiniao = ['bloco_d_estatais', 'bloco_d_tabelamento', 'bloco_e_punitivismo', 'bloco_e_educacao', 'bloco_g_corrupcao', 'bloco_g_pesquisa', 'bloco_g_politica_externa']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(cols_opiniao):
    sns.histplot(df[col], bins=5, kde=True, ax=axes[i], color='royalblue')
    axes[i].set_title(col)
    axes[i].set_xticks([1, 2, 3, 4, 5])

for j in range(len(cols_opiniao), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


## 4. Ponderação Amostral (Post-Stratification)
**Teoria**: Pesquisas de internet possuem viés de seleção. Normalmente amostras online têm excesso de jovens e alta renda em comparação ao eleitorado real aferido pelo TSE.

Para corrigir isso, atribuímos "pesos" a cada respondente. Se um jovem responde, seu peso será menor (pois já temos muitos); se um idoso responde, seu peso será maior (para equilibrar a balança).


In [ ]:
# Exemplo simulado de cálculo de peso amostral baseado em idade.
# Supondo que a população real (TSE) seja 20% jovens (16-24), mas nossa amostra tem 30%.

# Distribuição da Amostra
amostra_idade = df['idade'].value_counts(normalize=True)

# Distribuição Fictícia do TSE
tse_idade = {'16-24': 0.15, '25-34': 0.20, '35-44': 0.25, '45-59': 0.25, '60+': 0.15}

# Calculando pesos (TSE / Amostra)
pesos_idade = {k: tse_idade[k] / amostra_idade[k] for k in tse_idade.keys()}

df['peso_amostral'] = df['idade'].map(pesos_idade)
print('Pesos calculados por faixa etária:')
print(pesos_idade)
display(df[['idade', 'peso_amostral']].head())
